# 멘디 파손 분류 — 사진 단위 outsole / heel lining / 그 외 (Colab GPU)

검출 모델(YOLO26·RF-DETR)이 **박스**를 찾는 것과 달리, 이 노트북은 사진 한 장에
**outsole**(밑창 벌어짐) · **heel lining**(뒤꿈치 안감 손상) 파손이 **있는지**를 클래스마다 확률로 냅니다(다중 라벨).
둘 다 기준(0.5)보다 낮으면 **그 외**(두 부위 파손 없음)로 봅니다.

모델은 timm `convnext_tiny`(Apache-2.0, ImageNet-22k 사전학습)를 파인튜닝합니다. 512px · 15 epoch, T4에서 약 30~40분(추정).

**실행 순서**: 런타임 → 런타임 유형 변경 → **T4 GPU** → 런타임 → **모두 실행**. 3번 셀에서 파일을 올리라고 하면 한꺼번에 골라 올립니다.

**올릴 파일 3개** — YOLO 노트북과 같습니다 (이미 `MyDrive/mendii/yolo26_damage/`에 있으면 올리지 않아도 됩니다)
- CVAT 프로젝트 최종 export — `mendii_shoes_20261014_0900.zip`
- 밑창 재라벨 — `outsole_relabel.json`
- S3 읽기 전용 키 — `mendii-reader_accessKeys.csv`

**YOLO 노트북을 먼저 돌렸다면** 같은 Drive 폴더의 사진 캐시(`img1280.zip`)와 분리 목록(`runs/yolo26s_1024_final_e60/split.json`)을 그대로 씁니다.
그래서 학습·검증 사진이 YOLO와 같고, 8번 셀이 YOLO `last.pt`도 사진 단위로 다시 채점해 나란히 보여 줍니다.

**공정한 비교를 위해**: 학습 중에는 검증하지 않고 마지막 가중치로 채점합니다. 기준값(0.5)도 검증 사진으로 고르지 않습니다.
결과는 Drive `MyDrive/mendii/yolo26_damage/runs_cls/`에 저장됩니다.
사진에 작업지시서(개인정보)가 찍혀 있으니 결과 그림이 남은 노트북은 남에게 공유하지 마세요.

In [ ]:
# 1. 런타임 확인. CUDA가 False면 런타임 > 런타임 유형 변경에서 GPU(T4)를 고르고 다시 실행하세요.
import torch

print("torch", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU로는 학습이 사실상 불가능합니다. GPU 런타임으로 바꾸세요.")

In [ ]:
# 2. 설치. 버전을 고정해 두어야 나중에 다시 돌려도 같은 결과가 나옵니다.
# ultralytics는 YOLO 모델을 같은 코드로 다시 채점할 때만 씁니다.
!pip install -q "timm==1.0.15" "ultralytics==8.4.169" boto3

import importlib.metadata as md

print("timm", md.version("timm"), "| torch", md.version("torch"))

In [ ]:
# 3. 설정과 입력 파일. 바꿀 것은 이 셀 위쪽의 대문자 값뿐입니다.
DRIVE_DIR   = "MyDrive/mendii/yolo26_damage"   # 결과·사진 캐시를 저장할 Drive 폴더
BUCKET      = "mendii-sneakers"
REGION      = "ap-northeast-2"
STORE_SIZE  = 1280      # S3 원본(3456px)을 이 크기로 줄여 저장합니다.
REVIEW_GAP  = 100       # 박스 달린 사진 사이가 이 장수보다 멀면 그 사이는 아직 검수 안 한 것으로 봅니다.
SESSION_GAP = 300       # 촬영 간격이 이 초 이하면 같은 신발(같은 세션)로 봅니다.
VAL_RATIO   = 0.2       # 검증으로 뺄 세션 비율
NEG_PER_POS = 1.0       # 학습에 넣을 '파손 없음' 사진 수 = 파손 사진 수 × 이 값 (검증에는 전부 넣음)
REPEAT_OUTSOLE = 2      # outsole(밑창 벌어짐) 사진을 학습에 몇 번 넣을지. 드문 클래스라 2번 넣습니다(LVIS식 반복 샘플링).
SEED        = 0

# 학습. 안감 파손이 작아서(중앙값이 사진 한 변의 7%) 분류 모델 기본(224px)보다 크게 봅니다.
BACKBONE = "convnext_tiny.fb_in22k_ft_in1k"   # timm 모델 이름 (Apache-2.0). 더 크게: convnext_small.fb_in22k_ft_in1k
IMGSZ    = 512          # 640이면 조금 더 정확할 수 있지만 약 1.6배 느려집니다.
EPOCHS   = 15           # 8번 셀 전 7번 셀이 첫 epoch 시간을 보여 줍니다. 너무 길면 멈추고 줄이세요.
BATCH    = 16
LR       = 2e-4
THRESH   = 0.5          # 확률이 이 값 이상이면 그 부위 파손 있음. 검증 사진으로 고르지 않습니다.
NAME     = "cls_convnext_t_512_e15"   # 결과 폴더 이름. 설정을 바꿔 다시 학습하면 이름도 바꾸세요.
RESUME   = False        # True면 저장된 분리(split.json)를 다시 씁니다. 학습은 last.pt가 있으면 건너뜁니다.
YOLO_RUN = "yolo26s_1024_final_e60"   # 비교할 YOLO 학습(같은 Drive 폴더의 runs/). 그 분리 목록과 last.pt를 씁니다.
YOLO_IMGSZ = 1024
SPLIT_FROM = "yolo26s_1024"   # YOLO_RUN의 split.json이 없을 때만: YOLO 노트북과 같은 규칙으로 새로 나눔
QUICK_TEST = False      # True면 사진 120장·2 epoch로 노트북 전체가 도는지만 확인합니다.

# 다 검수했다고 확인된 CVAT job (첫 사진, 마지막 사진). 박스가 적어도 이 범위는 검수된 사진입니다.
KNOWN_REVIEWED = {
    "job 24": ("20240819_093048.jpg", "20240902_133744.jpg"),
    "job 25": ("20240903_092559.jpg", "20240913_154100.jpg"),
}

import csv
import io
import json
import os
import sys
import zipfile
from pathlib import Path

if QUICK_TEST:
    EPOCHS, IMGSZ, BATCH, NAME = 2, 320, 8, "quick_test"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content")
    OUT_DIR = Path("/content/drive") / DRIVE_DIR
else:                                   # Colab 밖(로컬 GPU 등)에서 돌릴 때
    ROOT = Path(os.environ.get("MENDII_ROOT", "mendii_yolo26")).resolve()
    OUT_DIR = ROOT / "out"
ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANN = "annotations/instances_default.json"


def read_cvat(p):
    """CVAT COCO export면 (export 시각, coco) 아니면 None."""
    try:
        with zipfile.ZipFile(p) as z:
            return z.getinfo(ANN).date_time, json.loads(z.read(ANN))
    except (zipfile.BadZipFile, KeyError, OSError):
        return None


def find_inputs():
    places = [ROOT, OUT_DIR]
    found = {}
    for p in sorted(q for d in places for q in d.glob("*.zip")):
        r = read_cvat(p)
        if r and p.name not in found:
            found[p.name] = (p, *r)
    keys = [p for d in places for p in d.glob("*.csv") if "accesskey" in p.name.lower()]
    return list(found.values()), (max(keys, key=lambda p: p.stat().st_mtime) if keys else None)


def find_relabel():
    ps = [p for d in (ROOT, OUT_DIR) for p in d.glob("outsole_relabel*.json")]
    return max(ps, key=lambda p: p.stat().st_mtime) if ps else None


def secret(name):
    if not IN_COLAB:
        return os.environ.get(name)
    from google.colab import userdata
    try:
        return userdata.get(name)
    except Exception:                    # 비밀이 없거나 이 노트북에 접근 권한을 안 준 경우
        return None


PROJECT_MIN_IMAGES = 5000   # 프로젝트 전체 export는 사진 18,168장, job export는 400여 장입니다.


def missing_inputs(exports, key_csv):
    miss = []
    if not any(len(e[2]["images"]) >= PROJECT_MIN_IMAGES for e in exports):
        miss.append("CVAT 프로젝트 최종 export (mendii_shoes_20261014_0900.zip, 사진 12,599장)")
    if not all(AWS_KEY) and key_csv is None:
        miss.append("S3 키 (mendii-reader_accessKeys.csv)")
    if find_relabel() is None:
        miss.append("밑창 재라벨 (outsole_relabel.json)")
    return miss


AWS_KEY = (secret("AWS_ACCESS_KEY_ID"), secret("AWS_SECRET_ACCESS_KEY"))
exports, key_csv = find_inputs()
for k in range(3):                       # 꼭 필요한 파일(프로젝트 export·키)이 올라올 때까지 다시 묻습니다.
    miss = missing_inputs(exports, key_csv)
    blocking = [m for m in miss if not m.startswith("밑창")]
    if not miss or not IN_COLAB or (k > 0 and not blocking):
        break
    from google.colab import files
    print("올릴 파일 (여러 개를 한꺼번에 골라도 됩니다):\n  - " + "\n  - ".join(miss))
    print("이미 있는 것:", [e[0].name for e in exports] or "없음")
    for name, data in files.upload().items():
        (ROOT / name).write_bytes(data)
    exports, key_csv = find_inputs()
miss = missing_inputs(exports, key_csv)
assert not [m for m in miss if not m.startswith("밑창")], "다음 파일이 없습니다: " + ", ".join(miss)
RELABEL = find_relabel()
print("밑창 재라벨:", RELABEL.name if RELABEL else "⚠️ 없음 — outsole은 CVAT 박스 그대로(기준이 섞여 있음)")

# 사진이 가장 많은 zip = 프로젝트 export(바탕). 그보다 나중에 받은 job export는 그 사진들의 박스를 바꿉니다.
main_path, main_time, coco = max(exports, key=lambda e: (len(e[2]["images"]), e[1]))
overrides = sorted((e for e in exports if e[0] != main_path and e[1] > main_time), key=lambda e: e[1])
ignored = [e[0].name for e in exports if e[0] != main_path and e[1] <= main_time]
fmt = lambda t: "%04d-%02d-%02d %02d:%02d" % t[:5]
print(f"바탕: {main_path.name} (export {fmt(main_time)} UTC) | 사진 {len(coco['images'])} | 박스 {len(coco['annotations'])}"
      f" | 라벨 {[c['name'] for c in coco['categories']]}")
for p, t, c in overrides:
    print(f"덮어쓰기: {p.name} (export {fmt(t)} UTC) | 사진 {len(c['images'])} | 박스 {len(c['annotations'])}")
if ignored:
    print("바탕보다 먼저 받은 export라 쓰지 않음:", ignored)
if not overrides and main_time < (2026, 10, 12, 17, 43, 0):
    print("⚠️ 10/12 이전 export입니다. job 24는 복원 전 상태일 수 있으니 job 24·25 수정본(10/12 export)을 같이 올리세요.")

if not all(AWS_KEY):
    assert key_csv is not None, "S3 키(mendii-reader_accessKeys.csv)를 찾지 못했습니다."
    row = next(csv.DictReader(io.StringIO(key_csv.read_text(encoding="utf-8-sig"))))
    AWS_KEY = (row["Access key ID"].strip(), row["Secret access key"].strip())
    if key_csv.parent == ROOT and IN_COLAB:
        key_csv.unlink()                 # 런타임 디스크에 키를 남기지 않습니다.

In [ ]:
# 4. 검수된 사진 고르기, job export로 덮어쓰기, COCO → YOLO 박스 변환.
from collections import Counter, defaultdict

CLASSES = ["outsole", "heel lining"]                      # YOLO 클래스 0, 1


def labels_by_name(c):
    """file_name -> [(라벨 이름, [x, y, w, h])]. 박스 없는 사진도 빈 목록으로 들어갑니다."""
    cat = {x["id"]: x["name"] for x in c["categories"]}
    ims = {im["id"]: im for im in c["images"]}
    out = {im["file_name"]: [] for im in c["images"]}
    for a in c["annotations"]:
        out[ims[a["image_id"]]["file_name"]].append((cat[a["category_id"]], a["bbox"]))
    return out


size = {im["file_name"]: (im["width"], im["height"]) for im in coco["images"]}
labels = labels_by_name(coco)

# 프로젝트 export의 사진 id 순서 = task·프레임 순서(대체로 촬영 순서). 박스가 달린 사진의 위치로 검수 구간을 추정합니다.
order = [im["file_name"] for im in sorted(coco["images"], key=lambda im: im["id"])]
hit = [k for k, n in enumerate(order) if labels[n]]
runs, start = [], hit[0]
for prev, k in zip(hit, hit[1:]):
    if k - prev > REVIEW_GAP:
        runs.append((start, prev))
        start = k
runs.append((start, hit[-1]))
reviewed = {n for s, e in runs for n in order[s:e + 1]}
for lo, hi in KNOWN_REVIEWED.values():
    reviewed |= {n for n in order if lo <= n <= hi}

# 나중에 받은 job export가 있으면 그 사진들의 박스를 통째로 바꾸고, 검수된 사진으로 봅니다.
for p, t, c in overrides:
    new = labels_by_name(c)
    changed = sum(sorted(map(str, labels.get(n, []))) != sorted(map(str, b)) for n, b in new.items())
    labels.update(new)
    size.update({im["file_name"]: (im["width"], im["height"]) for im in c["images"]})
    reviewed |= set(new)
    print(f"{p.name}: 사진 {len(new)}장 중 {changed}장의 박스가 바뀜")

# 밑창 재라벨: 그 사진들의 outsole 박스를 통째로 바꿉니다(빈 목록 = outsole 없음). heel lining 박스는 그대로 둡니다.
# 재라벨에서 '판단 불가'로 남긴 사진은 학습·채점에서 빼고, '파손 아님'으로 확인한 사진은 5번 셀에서 음성으로 먼저 씁니다.
unsure, hard_neg = set(), {}
if RELABEL:
    rl = json.loads(RELABEL.read_text())
    unsure = set(rl.get("uncertain", []))
    hard_neg = {f: c.split(":")[-1] for f, c in rl.get("category", {}).items() if c.split(":")[-1] in ("deleted", "none")}
    n_box = n_changed = 0
    for n, bs in rl["images"].items():
        if n not in size:
            continue
        W, H = size[n]
        old = sorted(round(v) for name, b in labels.get(n, []) if name == "outsole" for v in b)
        new = [("outsole", [x0 * W, y0 * H, (x1 - x0) * W, (y1 - y0) * H]) for x0, y0, x1, y1 in bs]
        labels[n] = [(name, b) for name, b in labels.get(n, []) if name != "outsole"] + new
        reviewed.add(n)
        n_box += len(new)
        n_changed += old != sorted(round(v) for _, b in new for v in b)
    print(f"{RELABEL.name}: 사진 {len(rl['images'])}장 → outsole 박스 {n_box}개 (바뀐 사진 {n_changed}장) | 기준: {rl.get('rule', '')}")

unknown = sorted({name for bs in labels.values() for name, _ in bs} - set(CLASSES) - {"not_sneakers"})
assert not unknown, f"모르는 라벨이 있습니다: {unknown}. CLASSES를 고치세요."

boxes = {}                                   # file_name -> [(cls, xc, yc, w, h)] (0~1 비율)
dropped = Counter()
for n in sorted(reviewed):
    W, H = size[n]
    if any(name == "not_sneakers" for name, _ in labels[n]):   # 운동화가 아닌 사진은 빼고 학습합니다.
        dropped["not_sneakers 사진"] += 1
        continue
    if n in unsure and not any(name == "outsole" for name, _ in labels[n]):
        dropped["판단 불가(재라벨 uncertain)"] += 1
        continue
    out = []
    for name, (x, y, w, h) in labels[n]:
        x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)
        if x1 - x0 < 4 or y1 - y0 < 4:
            dropped["너무 작은 박스"] += 1
            continue
        out.append((CLASSES.index(name), (x0 + x1) / 2 / W, (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H))
    boxes[n] = out

n_pos = sum(1 for b in boxes.values() if b)
print(f"\n검수 구간 {len(runs)}개 + 확인된 job → 검수된 사진 {len(boxes)}장 (파손 박스 있음 {n_pos} / 없음 {len(boxes) - n_pos})")
print(f"제외: 검수 안 된 사진 {len(size) - len(reviewed)}장, {dict(dropped)}")
print("월별 검수된 사진:", dict(sorted(Counter(n[:6] for n in boxes).items())))
print("박스 수:", {c: sum(b[0] == i for bs in boxes.values() for b in bs) for i, c in enumerate(CLASSES)})
for job, (lo, hi) in KNOWN_REVIEWED.items():    # CVAT 화면의 박스 수와 같은지 확인하세요.
    names = [n for n in boxes if lo <= n <= hi]
    print(f"{job}: 사진 {len(names)}장, 박스 {sum(len(boxes[n]) for n in names)}개")

In [ ]:
# 5. 세션 단위 train / val 분리. YOLO 학습의 split.json이 있으면 그대로 써서 YOLO와 같은 사진으로 학습·채점합니다.
# 세션을 (밑창 박스 있음, 안감 박스 있음) 네 종류로 나누고 종류마다 세션의 VAL_RATIO만큼을 val로 보냅니다.
# 그래야 적은 밑창 클래스도 val에 고르게 들어갑니다. 분리 목록은 Drive에 저장하고, RESUME 때는 그대로 다시 씁니다.
import random
from datetime import datetime


def taken_at(name):
    return datetime.strptime(name[:15], "%Y%m%d_%H%M%S")


# 세션은 검수 안 된 사진까지 포함한 전체 촬영 순서로 끊습니다.
all_names = sorted(size, key=lambda n: (taken_at(n), n))
session_of, sid, prev = {}, 0, None
for n in all_names:
    t = taken_at(n)
    if prev is None or (t - prev).total_seconds() > SESSION_GAP:
        sid += 1
    session_of[n] = sid
    prev = t

SPLIT_JSON = OUT_DIR / "runs_cls" / NAME / "split.json"
YOLO_SPLIT = OUT_DIR / "runs" / YOLO_RUN / "split.json"
inputs_now = [main_path.name] + [p.name for p, _, _ in overrides] + ([RELABEL.name] if RELABEL else [])
use_split = YOLO_SPLIT if (YOLO_SPLIT.exists() and not QUICK_TEST) else (SPLIT_JSON if RESUME and SPLIT_JSON.exists() else None)
if use_split:
    SPLIT_JSON = use_split
    saved = json.loads(SPLIT_JSON.read_text())
    train_names = [n for n in saved["train"] if n in boxes]
    val_names = [n for n in saved["val"] if n in boxes]
    print(f"저장된 분리를 다시 씁니다: {SPLIT_JSON}")
    if saved["inputs"] != inputs_now:
        print("⚠️ 처음 학습 때와 CVAT 파일이 다릅니다:", saved["inputs"], "→", inputs_now)
else:
    sessions = defaultdict(list)
    for n in boxes:
        sessions[session_of[n]].append(n)
    strata = defaultdict(list)
    for s, names in sessions.items():
        has = {b[0] for n in names for b in boxes[n]}
        strata[(0 in has, 1 in has)].append(s)

    rng = random.Random(SEED)
    prev_split = OUT_DIR / "runs" / SPLIT_FROM / "split.json"
    old_val, old_train = set(), set()
    if SPLIT_FROM and prev_split.exists() and not QUICK_TEST:
        prev = json.loads(prev_split.read_text())
        old_val = {session_of[n] for n in prev["val"] if n in session_of}
        old_train = {session_of[n] for n in prev["train"] if n in session_of} - old_val
    # 첫 학습의 검증 세션은 그대로 val, 첫 학습에 쓴 세션은 train, 나머지(새로 검수된 세션)만 층마다 VAL_RATIO씩 val로.
    val_sessions = set(old_val)
    for key in sorted(strata):
        ss = sorted(s for s in strata[key] if s not in old_val and s not in old_train)
        rng.shuffle(ss)
        val_sessions |= set(ss[:round(len(ss) * VAL_RATIO)])
    if old_val:
        print(f"{SPLIT_FROM}의 검증 세션 {len(old_val)}개 유지 + 새로 검수된 세션에서 {len(val_sessions) - len(old_val)}개 추가"
              f" (그 학습에 쓴 세션 {len(old_train)}개는 val에 넣지 않음)")

    val_names = sorted(n for n in boxes if session_of[n] in val_sessions)
    pos = sorted(n for n in boxes if session_of[n] not in val_sessions and boxes[n])
    neg_all = sorted(n for n in boxes if session_of[n] not in val_sessions and not boxes[n])
    n_neg = min(len(neg_all), round(len(pos) * NEG_PER_POS))
    # 음성은 재라벨에서 '파손 아님'으로 확인한 사진부터: 지운 오답(뒷굽 닳음·구멍 등) → 같은 신발의 다른 사진 → 나머지 무작위.
    tiers = [[n for n in neg_all if hard_neg.get(n) == "deleted"], [n for n in neg_all if hard_neg.get(n) == "none"],
             [n for n in neg_all if n not in hard_neg]]
    neg = []
    for tier in tiers:
        take = min(len(tier), n_neg - len(neg))
        neg += rng.sample(tier, take)
    neg = sorted(neg)
    print(f"음성 {len(neg)}장 = 지운 오답 {sum(hard_neg.get(n) == 'deleted' for n in neg)} + 같은 신발 확인 "
          f"{sum(hard_neg.get(n) == 'none' for n in neg)} + 무작위 {sum(n not in hard_neg for n in neg)}")
    if QUICK_TEST:
        pos, neg = pos[:90], neg[:30]
        val_names = [n for n in val_names if boxes[n]][:40] + [n for n in val_names if not boxes[n]][:20]
    train_names = sorted(pos + neg)
    print(f"세션 {len(sessions)}개 → val {len(val_sessions)}개 (train 파손 없음 후보 {len(neg_all)}장 중 {len(neg)}장 뽑음)")
    SPLIT_JSON.parent.mkdir(parents=True, exist_ok=True)
    SPLIT_JSON.write_text(json.dumps({"inputs": inputs_now, "train": train_names, "val": val_names}))

split = {**{n: "train" for n in train_names}, **{n: "val" for n in val_names}}
assert not {session_of[n] for n in train_names} & {session_of[n] for n in val_names}, "같은 세션이 train과 val에 섞였습니다."

print(f"\n{'':<14}{'train':>8}{'val':>8}")
print(f"{'사진':<14}{len(train_names):>8}{len(val_names):>8}")
print(f"{'  파손 없음':<14}{sum(not boxes[n] for n in train_names):>8}{sum(not boxes[n] for n in val_names):>8}")
for i, c in enumerate(CLASSES):
    ti = sum(any(b[0] == i for b in boxes[n]) for n in train_names)
    vi = sum(any(b[0] == i for b in boxes[n]) for n in val_names)
    tb = sum(b[0] == i for n in train_names for b in boxes[n])
    vb = sum(b[0] == i for n in val_names for b in boxes[n])
    print(f"{c:<14}{ti:>8}{vi:>8}   사진 / 박스 {tb} · {vb}")

In [ ]:
# 6. S3 원본을 받아 STORE_SIZE로 줄여 저장. 한 번 받은 사진은 Drive의 zip 한 개로 저장해 다음부터 다시 받지 않습니다.
import shutil
import time
from concurrent.futures import ThreadPoolExecutor

import boto3
from botocore.config import Config
from PIL import Image, ImageOps

IMG_DIR = ROOT / f"img{STORE_SIZE}"
IMG_DIR.mkdir(exist_ok=True)
CACHE_ZIP = OUT_DIR / f"img{STORE_SIZE}.zip"

need = sorted(split)
have = {p.name for p in IMG_DIR.glob("*.jpg")}
if CACHE_ZIP.exists() and not set(need) <= have:
    t0 = time.time()
    local = ROOT / CACHE_ZIP.name
    shutil.copy(CACHE_ZIP, local)            # Drive에서 바로 풀면 느려서 로컬로 복사한 뒤 풉니다.
    try:
        with zipfile.ZipFile(local) as z:
            z.extractall(IMG_DIR)
    except zipfile.BadZipFile:
        print("Drive 캐시 zip이 깨져 있어 S3에서 다시 받습니다.")
    local.unlink()
    have = {p.name for p in IMG_DIR.glob("*.jpg")}
    print(f"Drive 캐시에서 {len(have)}장 ({time.time() - t0:.0f}초)")

todo = [n for n in need if n not in have]
s3 = boto3.client("s3", region_name=REGION, aws_access_key_id=AWS_KEY[0], aws_secret_access_key=AWS_KEY[1],
                  config=Config(max_pool_connections=32, retries={"max_attempts": 8, "mode": "adaptive"}))


def fetch(name):
    body = s3.get_object(Bucket=BUCKET, Key=name)["Body"].read()
    im = Image.open(io.BytesIO(body))
    im.draft("RGB", (STORE_SIZE, STORE_SIZE))   # JPEG를 1/2 크기로 바로 풀어 빠르게 줄입니다.
    im = ImageOps.exif_transpose(im).convert("RGB")
    im.thumbnail((STORE_SIZE, STORE_SIZE), Image.LANCZOS)
    tmp = IMG_DIR / (name + ".part")
    im.save(tmp, "JPEG", quality=90)
    tmp.rename(IMG_DIR / name)
    return len(body)


if todo:
    t0, done, nbytes, failed = time.time(), 0, 0, []
    with ThreadPoolExecutor(24) as ex:
        futs = {ex.submit(fetch, n): n for n in todo}
        for f in futs:
            try:
                nbytes += f.result()
            except Exception as e:           # 받기 실패한 사진은 학습에서 뺍니다.
                failed.append((futs[f], repr(e)[:120]))
            done += 1
            if done % 500 == 0 or done == len(todo):
                el = time.time() - t0
                print(f"  {done}/{len(todo)}장 {nbytes / 1e9:.1f}GB {el:.0f}초 (남은 시간 약 {el / done * (len(todo) - done) / 60:.0f}분)")
    if failed:
        print(f"받기 실패 {len(failed)}장 — 학습에서 뺍니다. 예:", failed[:3])
        assert len(failed) < 0.05 * len(todo), "실패가 너무 많습니다. 키·인터넷을 확인하고 이 셀을 다시 실행하세요."
        for n, _ in failed:
            split.pop(n, None)
    if not QUICK_TEST and len(failed) < len(todo):
        # 캐시 zip 다시 쓰기 (사진은 이미 JPEG라 압축 없이 묶기만). 다 쓴 뒤에 이름을 바꿔 깨진 zip이 남지 않게 합니다.
        tmp_zip = ROOT / "cache.zip"
        with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_STORED) as z:
            for p in sorted(IMG_DIR.glob("*.jpg")):
                z.write(p, p.name)
        part = CACHE_ZIP.with_name(CACHE_ZIP.name + ".part")
        shutil.copy(tmp_zip, part)
        os.replace(part, CACHE_ZIP)
        tmp_zip.unlink()
        print("Drive 캐시 저장:", CACHE_ZIP, f"{CACHE_ZIP.stat().st_size / 1e9:.2f}GB")

train_names = [n for n in train_names if n in split]
val_names = [n for n in val_names if n in split]
print(f"준비 완료: train {len(train_names)}장 / val {len(val_names)}장")

In [ ]:
# 7. 사진 단위 정답과 학습.
# 정답 = 그 사진에 클래스별 박스가 하나라도 있는지(0/1). 파손 없는 사진은 [0, 0].
# outsole 사진은 YOLO 노트북처럼 학습에 REPEAT_OUTSOLE번 넣습니다(드문 클래스 반복 샘플링).
# 학습 중에는 검증하지 않고 마지막 가중치(last.pt)를 씁니다. 끊기면 다시 실행하면 처음부터 학습합니다.
import math
import time

import numpy as np
import timm
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

RUN = OUT_DIR / "runs_cls" / NAME
RUN.mkdir(parents=True, exist_ok=True)
FINAL = RUN / "last.pt"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)


def target(n):
    has = {b[0] for b in boxes[n]}
    return [float(i in has) for i in range(len(CLASSES))]


train_items = list(train_names) + [n for n in train_names if any(b[0] == 0 for b in boxes[n])
                                   for _ in range(REPEAT_OUTSOLE - 1)]
if QUICK_TEST:
    train_items = train_items[:120]
    EPOCHS = 2
print(f"학습 사진 {len(train_items)}장 (outsole 반복 포함) | "
      + " · ".join(f"{c} {int(sum(target(n)[i] for n in train_items))}" for i, c in enumerate(CLASSES)))

# 크게 자르면 작은 안감 파손이 잘려 나가므로 자르기는 85% 이상만 합니다. 좌우 뒤집기는 신발 좌우 짝과 같아 안전합니다.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMGSZ, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.25, 0.25, 0.15, 0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize((IMGSZ, IMGSZ)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])


class Photos(Dataset):
    def __init__(self, names, tf):
        self.names, self.tf = names, tf

    def __len__(self):
        return len(self.names)

    def __getitem__(self, i):
        n = self.names[i]
        with Image.open(IMG_DIR / n) as im:
            x = self.tf(im.convert("RGB"))
        return x, torch.tensor(target(n))


def make_model(pretrained):
    return timm.create_model(BACKBONE, pretrained=pretrained, num_classes=len(CLASSES))


if FINAL.exists() and not QUICK_TEST:
    print("이미 끝난 학습입니다. 그대로 8번부터 채점합니다:", FINAL)
else:
    torch.manual_seed(SEED)
    dl = DataLoader(Photos(train_items, train_tf), batch_size=BATCH, shuffle=True, num_workers=2,
                    pin_memory=True, drop_last=True)
    net = make_model(True).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=0.05)
    total, warm = EPOCHS * len(dl), len(dl)          # 첫 epoch 동안 lr을 서서히 올린 뒤 코사인으로 내립니다.
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    loss_fn = nn.BCEWithLogitsLoss()

    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        net.train()
        run_loss, te = 0.0, time.time()
        for x, t in dl:
            x, t = x.to(DEVICE, non_blocking=True), t.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
                loss = loss_fn(net(x), t)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            run_loss += loss.item() * len(x)
        msg = f"epoch {ep}/{EPOCHS}  loss {run_loss / (len(dl) * BATCH):.4f}  {time.time() - te:.0f}초"
        if ep == 1:
            msg += f"  (전체 예상 {(time.time() - te) * EPOCHS / 60:.0f}분)"
        print(msg)

    torch.save({"model": BACKBONE, "classes": CLASSES, "img_size": IMGSZ, "multilabel": True,
                "threshold": THRESH, "state_dict": net.state_dict()}, FINAL)
    print(f"학습 {(time.time() - t0) / 60:.0f}분 | 모델: {FINAL}")
    del net, opt

In [ ]:
# 8. 같은 검증 사진 · 같은 정답으로 분류 모델과 YOLO26을 '사진 단위'로 채점합니다.
#  - 분류 모델 점수 = 클래스별 확률, YOLO 점수 = 그 사진에서 그 클래스 박스의 최고 conf (박스 없으면 0)
#  - AP / AUROC: 기준값 없이 순서만 보는 점수(0~1). 기준값을 바꿔도 변하지 않아 두 모델을 공정하게 비교합니다.
#  - 찾은 비율 @ 잘못 경보 10%: 파손 없는 사진의 10%에만 경보가 나도록 기준을 맞췄을 때 파손 사진을 찾은 비율
#  - 신발(세션) 단위: 세션 점수 = 그 세션 사진들의 최고 점수. 한 장이라도 찾으면 찾은 것(앱이 판단하는 방식)
#  - 차이의 90% 구간: 검증 세션을 1,000번 다시 뽑아(세션 단위 부트스트랩) 계산. 0을 걸치면 우연일 수 있는 차이입니다.
import gc

import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

gc.collect()
torch.cuda.empty_cache()
VAL = sorted(val_names)
Y = np.array([target(n) for n in VAL])                 # [사진, 클래스] 정답 0/1


@torch.no_grad()
def predict_cls(names):
    ck = torch.load(FINAL, map_location="cpu", weights_only=False)
    net = make_model(False)
    net.load_state_dict(ck["state_dict"])
    net.to(DEVICE).eval()
    dl = DataLoader(Photos(names, eval_tf), batch_size=32, num_workers=2)
    out = []
    for x, _ in dl:
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=DEVICE == "cuda"):
            out.append(torch.sigmoid(net(x.to(DEVICE)).float()).cpu())
    del net
    return torch.cat(out).numpy()


def predict_yolo(names, weights):
    from ultralytics import YOLO
    net = YOLO(str(weights))
    s = np.zeros((len(names), len(CLASSES)))
    idx = {n: i for i, n in enumerate(names)}
    for i in range(0, len(names), 8):
        chunk = [str(IMG_DIR / n) for n in names[i:i + 8]]
        for res in net.predict(chunk, imgsz=YOLO_IMGSZ, conf=0.01, verbose=False):
            k = idx[Path(res.path).name]
            for c, conf in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist()):
                s[k, int(c)] = max(s[k, int(c)], conf)
    del net
    return s


def recall_at_fa(y, s, fa=0.10):
    neg = np.sort(s[y == 0])[::-1]
    if len(neg) == 0 or y.sum() == 0:
        return float("nan")
    thr = neg[min(len(neg) - 1, int(np.floor(fa * len(neg))))]
    return float((s[y == 1] > thr).mean())


def safe(f, y, s):
    return float(f(y, s)) if 0 < y.sum() < len(y) else float("nan")


SESS = np.array([session_of[n] for n in VAL])


def by_session(y, s):
    keys = np.unique(SESS)
    return (np.array([y[SESS == k].max() for k in keys]), np.array([s[SESS == k].max() for k in keys]))


def table(S, name):
    rows = []
    cols = [(c, Y[:, i], S[:, i]) for i, c in enumerate(CLASSES)] + [("파손 있음(둘 중 하나)", Y.max(1), S.max(1))]
    for c, y, s in cols:
        ys, ss = by_session(y, s)
        rows.append({"모델": name, "클래스": c, "검증 파손 사진": int(y.sum()),
                     "AP": safe(average_precision_score, y, s), "AUROC": safe(roc_auc_score, y, s),
                     "찾음@경보10%": recall_at_fa(y, s),
                     "신발 AP": safe(average_precision_score, ys, ss), "신발 찾음@경보10%": recall_at_fa(ys, ss)})
    return rows


S_cls = predict_cls(VAL)
results = {"cls": S_cls}
yolo_w = OUT_DIR / "runs" / YOLO_RUN / "weights" / "last.pt"
if yolo_w.exists() and not QUICK_TEST:
    results["yolo"] = predict_yolo(VAL, yolo_w)
else:
    print("YOLO last.pt가 없어 분류 모델만 채점합니다:", yolo_w)

df = pd.DataFrame([r for k, S in results.items() for r in table(S, {"cls": NAME, "yolo": YOLO_RUN}[k])])
pd.set_option("display.float_format", "{:.3f}".format)
display(df)

# 앱 관점 3갈래(outsole / heel lining / 그 외) — THRESH 기준. 두 부위가 다 있는 사진은 확률이 높은 쪽을 맞힌 것으로 봅니다.
def three_way(y_row, s_row):
    truth = "그 외" if y_row.max() == 0 else None
    pred = "그 외" if s_row.max() < THRESH else CLASSES[int(s_row.argmax())]
    if truth is None:
        truth = pred if pred in [c for i, c in enumerate(CLASSES) if y_row[i]] else CLASSES[int(np.argmax(y_row))]
    return truth, pred


tw = pd.DataFrame([three_way(Y[i], S_cls[i]) for i in range(len(VAL))], columns=["정답", "예측"])
print(f"\n3갈래 정확도 {(tw['정답'] == tw['예측']).mean():.3f} (THRESH = {THRESH})")
display(pd.crosstab(tw["정답"], tw["예측"]))

# 세션 부트스트랩: 분류 AP − YOLO AP의 90% 구간
boot = {}
if "yolo" in results:
    rng = np.random.default_rng(SEED)
    keys = np.unique(SESS)
    members = {k: np.where(SESS == k)[0] for k in keys}
    for i, c in enumerate(CLASSES):
        diffs = []
        for _ in range(1000):
            ix = np.concatenate([members[k] for k in rng.choice(keys, len(keys), replace=True)])
            y = Y[ix, i]
            if 0 < y.sum() < len(y):
                diffs.append(average_precision_score(y, results["cls"][ix, i]) - average_precision_score(y, results["yolo"][ix, i]))
        lo, hi = np.percentile(diffs, [5, 95])
        boot[c] = [float(lo), float(hi)]
        side = "분류가 낫다" if lo > 0 else ("YOLO가 낫다" if hi < 0 else "0을 걸침(우연일 수 있음)")
        print(f"{c}: 사진 AP 차이(분류 − YOLO) 90% 구간 [{lo:+.3f}, {hi:+.3f}] → {side}")

summary = {"name": NAME, "backbone": BACKBONE, "imgsz": IMGSZ, "epochs": EPOCHS, "threshold": THRESH,
           "split": str(SPLIT_JSON), "n_val": len(VAL), "table": df.to_dict("records"),
           "three_way_acc": float((tw["정답"] == tw["예측"]).mean()), "ap_diff_90ci": boot}
(RUN / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print("\n저장:", RUN / "summary.json")
print(json.dumps({k: summary[k] for k in ("name", "n_val", "three_way_acc", "ap_diff_90ci")}, ensure_ascii=False))

In [ ]:
# 9. 분류 모델이 놓친 사진 / 잘못 경보한 사진 (각 클래스 상위 8장). 제목은 Colab 폰트에 한글이 없어 영어로 씁니다.
import matplotlib.pyplot as plt


def grid(names, title):
    if not names:
        print("없음:", title)
        return
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    for ax in axes.flat:
        ax.axis("off")
    for ax, n in zip(axes.flat, names):
        with Image.open(IMG_DIR / n) as im:
            ax.imshow(im.convert("RGB"))
        k = VAL.index(n)
        ax.set_title(f"{n}\n" + " ".join(f"{c[:4]} {S_cls[k, i]:.2f}" for i, c in enumerate(CLASSES)), fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


for i, c in enumerate(CLASSES):
    missed = sorted([n for k, n in enumerate(VAL) if Y[k, i] and S_cls[k, i] < THRESH], key=lambda n: S_cls[VAL.index(n), i])
    alarm = sorted([n for k, n in enumerate(VAL) if not Y[k, i] and S_cls[k, i] >= THRESH], key=lambda n: -S_cls[VAL.index(n), i])
    print(f"{c}: 놓침 {len(missed)}장 / 잘못 경보 {len(alarm)}장")
    grid(missed[:8], f"{c}: missed")
    grid(alarm[:8], f"{c}: false alarm (check: unlabeled damage?)")

## 결과 읽는 법

- **8번 표**의 `AP`가 사진 단위 핵심 점수입니다(0~1, 높을수록 좋음). `신발 AP`는 세션(신발 한 켤레) 단위입니다.
- YOLO 줄은 박스 모델을 "그 사진에서 가장 확신한 박스 점수"로 바꿔 같은 방식으로 채점한 것입니다.
  박스 위치는 보지 않으므로, **위치가 필요 없는 '이 사진에 어떤 수선이 필요한가' 판단만** 비교합니다.
- **결정 규칙**: 사진 단위 AP 차이의 90% 구간이 0을 걸치면 성능이 비슷한 것으로 보고, 위치까지 주는 검출 모델을 씁니다.
  분류 모델이 확실히 이길 때(구간 전체가 +)만 앱의 1차 필터로 검토합니다.
- 서빙: `damage_model/cls_serving/`의 `serve.py --ckpt last.pt`로 같은 가중치를 API로 띄우고 `shoe_app.py`로 시험합니다.